# Part A — Graph-Based Search (Member 1)

**This section is fully independent of the CSP section below.**
It only needs an `Environment*.txt` file. It does **not** import or call any CSP
functions (`backtracking_search`, `load_schedule`, etc.).

**Source of truth:** the Python package (`core/`, API, pytest) loads *these*
Part A cells via `notebook_cells.part_a`, so notebook / library / GUI stay in sync.

## Problem formulation
- **State:** robot position `(row, col)` — zero-based; top-left is `(0,0)`.
- **Initial state:** the `start` argument passed to `search`.
- **Actions:** UP, DOWN, LEFT, RIGHT (no diagonals). Valid only if the next cell
  is inside the grid and is not an obstacle.
- **Transition model:** a valid action moves the robot to that neighbour.
- **Goal state:** the `goal` argument passed to `search`.
- **Step cost:** every valid move costs `1`.

Dirty labels `D1..Dn` follow the order of dirty coordinates in the environment file.


## Imports for graph search

Standard library only — no Qt, no CSP, no integration modules.


In [ ]:
from __future__ import annotations

import re
import time
from collections import deque
from dataclasses import dataclass, field
from heapq import heappop, heappush
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple, Union

# A grid cell is always (row, col), zero-based.
Position = Tuple[int, int]


## Environment file format

One logical item per line (blank lines are ignored):

1. `rows,cols` — e.g. `8,8` or `10,10` or `16,16` (this decides the grid size)
2. number of obstacles
3. obstacle coordinates as `(row,col)` tuples
4. number of dirty locations
5. dirty coordinates (file order = D1, D2, …)
6. charging station as one `(row,col)` tuple


In [ ]:
class EnvironmentFormatError(Exception):
    """Raised when an environment file is malformed (wrong syntax / structure)."""

    def __init__(self, message: str, line_number: Optional[int] = None) -> None:
        self.line_number = line_number
        # Include the line number in the message so debugging is easy in viva.
        full = f"Line {line_number}: {message}" if line_number is not None else message
        super().__init__(full)


# Matches coordinates like (3, 7) with optional spaces.
_COORD_RE = re.compile(r"\(\s*(-?\d+)\s*,\s*(-?\d+)\s*\)")


@dataclass
class Grid:
    """Grid-based vacuum environment used by graph search.

    Attributes:
        rows, cols: grid size (from line 1 of the file).
        obstacles: cells the robot cannot enter.
        dirty_locations: ordered list — index 0 is D1, index 1 is D2, …
        charging_station: start/end cell for a full mission (used here as a landmark).
    """

    rows: int
    cols: int
    obstacles: Set[Position] = field(default_factory=set)
    dirty_locations: List[Position] = field(default_factory=list)
    charging_station: Position = (0, 0)

    def is_in_bounds(self, pos: Position) -> bool:
        """True if pos lies inside [0..rows) x [0..cols)."""
        r, c = pos
        return 0 <= r < self.rows and 0 <= c < self.cols

    def is_obstacle(self, pos: Position) -> bool:
        """True if pos is blocked."""
        return pos in self.obstacles

    def is_free(self, pos: Position) -> bool:
        """Walkable cell: in bounds and not an obstacle.

        Dirty cells and the charging station are walkable.
        """
        return self.is_in_bounds(pos) and not self.is_obstacle(pos)

    def task_label(self, index: int) -> str:
        """Map dirty index 0 -> 'D1', 1 -> 'D2', …"""
        return f"D{index + 1}"

    def label_to_position(self, label: str) -> Position:
        """Map 'D1'/'C'/'Station' to a coordinate."""
        normalized = label.strip().upper()
        if normalized in {"C", "STATION", "CHARGING", "CHARGING_STATION"}:
            return self.charging_station
        match = re.fullmatch(r"D(\d+)", normalized)
        if match:
            idx = int(match.group(1)) - 1
            if 0 <= idx < len(self.dirty_locations):
                return self.dirty_locations[idx]
        raise KeyError(f"Unknown label: {label}")

    def position_to_label(self, pos: Position) -> Optional[str]:
        """Return 'C', 'Dk', or None."""
        if pos == self.charging_station:
            return "C"
        for i, dirty in enumerate(self.dirty_locations):
            if dirty == pos:
                return self.task_label(i)
        return None

    def cell_type(self, pos: Position) -> str:
        """Single-character cell type for printing: C / D / X / ."""
        if pos in self.obstacles:
            return "X"
        if pos == self.charging_station:
            return "C"
        if pos in self.dirty_locations:
            return "D"
        return "."

    def to_matrix(self) -> List[List[str]]:
        """2D list of cell characters for GUI / tests ('C','D','X','.')."""
        return [[self.cell_type((r, c)) for c in range(self.cols)] for r in range(self.rows)]

    def render_text(
        self,
        path: Optional[Sequence[Position]] = None,
        cleaned: Optional[Iterable[Position]] = None,
    ) -> str:
        """ASCII drawing of the grid; path cells marked with *."""
        cleaned_set = set(cleaned or [])
        path_set = set(path or [])
        lines = ["   " + " ".join(str(c) for c in range(self.cols))]
        for r in range(self.rows):
            cells = []
            for c in range(self.cols):
                pos = (r, c)
                ch = self.cell_type(pos)
                if pos in cleaned_set and ch == "D":
                    ch = "+"
                elif pos in path_set and ch == ".":
                    ch = "*"
                label = self.position_to_label(pos)
                if label and label.startswith("D") and pos not in cleaned_set:
                    cells.append(f"{label:>2}")
                else:
                    cells.append(f"{ch:>2}")
            lines.append(f"{r:2} " + " ".join(cells))
        return "\n".join(lines)


def _nonempty_lines(text: str) -> List[Tuple[int, str]]:
    """Keep only non-blank lines as (1-based line number, stripped text)."""
    return [(i, raw.strip()) for i, raw in enumerate(text.splitlines(), 1) if raw.strip()]


def _parse_coords(text: str, line_number: int, expected: Optional[int] = None) -> List[Position]:
    """Extract all (row,col) tuples from one line."""
    coords = [(int(m.group(1)), int(m.group(2))) for m in _COORD_RE.finditer(text)]
    residual = re.sub(r"[\s,]+", "", _COORD_RE.sub("", text))
    if residual:
        raise EnvironmentFormatError(f"Unexpected characters: {residual!r}", line_number)
    if expected is not None and len(coords) != expected:
        raise EnvironmentFormatError(
            f"Expected {expected} coordinate(s), found {len(coords)}", line_number
        )
    return coords


def load_environment(filepath: Union[str, Path]) -> Grid:
    """Parse an Environment.txt-style file into a Grid.

    The first line (`rows,cols`) decides whether the map is 8x8, 10x10, 16x16, etc.
    """
    text = Path(filepath).read_text(encoding="utf-8")
    lines = _nonempty_lines(text)
    if len(lines) < 5:
        raise EnvironmentFormatError("File too short")

    idx = 0
    line_no, content = lines[idx]
    dim = re.fullmatch(r"(\d+)\s*,\s*(\d+)", content)
    if not dim:
        raise EnvironmentFormatError(f"Expected rows,cols, got {content!r}", line_no)
    rows, cols = int(dim.group(1)), int(dim.group(2))
    idx += 1

    line_no, content = lines[idx]
    n_obstacles = int(content)
    idx += 1

    obstacles: List[Position] = []
    if n_obstacles == 0:
        # Empty obstacle list is allowed when the count is 0.
        if idx < len(lines) and _COORD_RE.search(lines[idx][1]):
            obstacles = _parse_coords(lines[idx][1], lines[idx][0], expected=0)
            idx += 1
    else:
        obstacles = _parse_coords(lines[idx][1], lines[idx][0], expected=n_obstacles)
        idx += 1

    line_no, content = lines[idx]
    n_dirty = int(content)
    idx += 1
    dirty = _parse_coords(lines[idx][1], lines[idx][0], expected=n_dirty)
    idx += 1
    station = _parse_coords(lines[idx][1], lines[idx][0], expected=1)[0]

    return Grid(rows, cols, set(obstacles), list(dirty), station)


## Required search helpers

Course-required functions with exact roles:

- `get_successors` — legal moves UP, DOWN, LEFT, RIGHT (deterministic order)
- `goal_test` — have we reached the goal?
- `path_cost` — number of moves = `len(path) - 1`
- `heuristic` — Manhattan distance (admissible + consistent for 4-dir unit cost)

**Why Manhattan is appropriate:** with only UP/DOWN/LEFT/RIGHT and cost 1 per step,
Manhattan distance is the shortest path on an empty grid. Obstacles can only make
real paths longer, so `h` never overestimates true cost (admissible). It is also
consistent, so A* with a closed set stays optimal.


In [ ]:
# Deterministic expansion order required by the project handout.
_ACTIONS = (("UP", -1, 0), ("DOWN", 1, 0), ("LEFT", 0, -1), ("RIGHT", 0, 1))
SUPPORTED_ALGORITHMS = ("bfs", "astar", "ucs", "greedy")


def get_successors(state, grid):
    """Return [(next_state, action, step_cost), ...] in UP/DOWN/LEFT/RIGHT order.

    Excludes out-of-bounds cells and obstacles. step_cost is always 1.
    """
    successors = []
    r, c = state
    for action, dr, dc in _ACTIONS:
        nxt = (r + dr, c + dc)
        if grid.is_free(nxt):
            successors.append((nxt, action, 1))
    return successors


def goal_test(state, goal_state):
    """True when the current state equals the goal."""
    return state == goal_state


def path_cost(path):
    """Total path cost = number of moves = len(path)-1.

    Empty / None / single-cell paths return 0.
    """
    if not path:
        return 0
    return max(0, len(path) - 1)


def heuristic(state, goal_state):
    """Manhattan distance |dr| + |dc| (admissible & consistent)."""
    return abs(state[0] - goal_state[0]) + abs(state[1] - goal_state[1])


def reconstruct_path(parent, goal):
    """Follow parent pointers backwards from goal to rebuild the start→goal path."""
    path = [goal]
    current = goal
    while parent.get(current) is not None:
        current = parent[current]
        path.append(current)
    path.reverse()
    return path


## BFS and A* (graph search)

Both algorithms:
- count a node as **expanded** when it is **popped** and successors are generated
- run the **goal test on pop** (same convention so node counts are comparable)

Optional extras: UCS (order by `g`) and Greedy (order by `h`).


In [ ]:
@dataclass
class SearchResult:
    """Full search trace (path + stats) for printing / comparison."""

    path: Optional[List[Position]]
    cost: Optional[int]
    nodes_expanded: int
    expanded_order: List[Position] = field(default_factory=list)
    frontier_max_size: int = 0
    runtime_seconds: float = 0.0
    algorithm: str = ""
    found: bool = False
    message: str = ""


def _normalize_algorithm(algorithm: str) -> str:
    """Accept 'bfs', 'astar', 'A*', 'ucs', 'greedy' (case-insensitive)."""
    key = algorithm.strip().lower().replace(" ", "").replace("-", "").replace("_", "")
    aliases = {
        "bfs": "bfs", "astar": "astar", "a*": "astar",
        "ucs": "ucs", "uniformcost": "ucs", "uniformcostsearch": "ucs",
        "greedy": "greedy", "greedybestfirst": "greedy",
    }
    if key not in aliases:
        raise ValueError(f"Unknown algorithm {algorithm!r}. Supported: {SUPPORTED_ALGORITHMS}")
    return aliases[key]


def _bfs(grid, start, goal):
    """Breadth-first search (uninformed, optimal for unit step costs).

    FIFO queue + parent map. Nodes are not enqueued twice (`nxt not in parent`).
    """
    frontier = deque([start])
    parent = {start: None}  # also acts as the 'seen' set
    expanded_order = []
    frontier_max = 1
    nodes_expanded = 0
    while frontier:
        frontier_max = max(frontier_max, len(frontier))
        current = frontier.popleft()
        nodes_expanded += 1
        expanded_order.append(current)
        if goal_test(current, goal):
            path = reconstruct_path(parent, goal)
            return SearchResult(
                path, path_cost(path), nodes_expanded, expanded_order,
                frontier_max, 0.0, "bfs", True, "Path found",
            )
        for nxt, _, _ in get_successors(current, grid):
            if nxt not in parent:
                parent[nxt] = current
                frontier.append(nxt)
    return SearchResult(
        None, None, nodes_expanded, expanded_order, frontier_max,
        0.0, "bfs", False, "No path found",
    )


def _best_first(grid, start, goal, algorithm):
    """Shared priority-queue search for A* / UCS / Greedy.

    Priority key is (primary, secondary, insertion_counter) for deterministic ties.
    Stale heap entries are skipped via best_g / closed set.
    """
    counter = 0
    heap = []

    def priority(g, state):
        h = heuristic(state, goal)
        if algorithm == "astar":
            return (g + h, h, counter)   # f = g + h
        if algorithm == "ucs":
            return (g, h, counter)       # optional: uniform-cost
        return (h, g, counter)           # optional: greedy best-first

    heappush(heap, (priority(0, start), counter, start, 0))
    counter += 1
    parent = {start: None}
    best_g = {start: 0}
    closed = set()
    expanded_order = []
    frontier_max = 1
    nodes_expanded = 0
    in_frontier = {start}

    while heap:
        frontier_max = max(frontier_max, len(in_frontier))
        _, _, current, g = heappop(heap)
        in_frontier.discard(current)
        if current in closed or g > best_g.get(current, g):
            continue  # stale entry
        nodes_expanded += 1
        expanded_order.append(current)
        closed.add(current)
        if goal_test(current, goal):
            path = reconstruct_path(parent, goal)
            return SearchResult(
                path, path_cost(path), nodes_expanded, expanded_order,
                frontier_max, 0.0, algorithm, True, "Path found",
            )
        for nxt, _, step in get_successors(current, grid):
            if nxt in closed:
                continue
            new_g = g + step
            if new_g < best_g.get(nxt, float("inf")):
                best_g[nxt] = new_g
                parent[nxt] = current
                heappush(heap, (priority(new_g, nxt), counter, nxt, new_g))
                counter += 1
                in_frontier.add(nxt)
    return SearchResult(
        None, None, nodes_expanded, expanded_order, frontier_max,
        0.0, algorithm, False, "No path found",
    )


def search_with_trace(grid, start, goal, algorithm):
    """Run a search and return a full SearchResult (used for demos / comparison)."""
    algo = _normalize_algorithm(algorithm)
    t0 = time.perf_counter()
    # Invalid endpoints: fail softly (no crash) with zero expansions.
    if not grid.is_in_bounds(start) or grid.is_obstacle(start):
        return SearchResult(None, None, 0, [], 0, time.perf_counter() - t0, algo, False,
                            f"Invalid start {start}")
    if not grid.is_in_bounds(goal) or grid.is_obstacle(goal):
        return SearchResult(None, None, 0, [], 0, time.perf_counter() - t0, algo, False,
                            f"Invalid goal {goal}")
    # Already at the goal → ([start], 0, 0).
    if start == goal:
        return SearchResult([start], 0, 0, [], 0, time.perf_counter() - t0, algo, True,
                            "Start equals goal (zero expansions)")
    result = _bfs(grid, start, goal) if algo == "bfs" else _best_first(grid, start, goal, algo)
    result.runtime_seconds = time.perf_counter() - t0
    return result


def search(grid, start, goal, algorithm):
    """Course-required API: return (path, cost, nodes_expanded).

    - path found → list of (row,col) from start to goal inclusive
    - no path → (None, None, nodes_expanded)
    - start == goal → ([start], 0, 0)
    """
    result = search_with_trace(grid, start, goal, algorithm)
    return result.path, result.cost, result.nodes_expanded


def format_comparison_table(results):
    """Pretty text table for BFS vs A* (and optional others)."""
    headers = ("Algorithm", "Path found", "Cost", "Nodes expanded", "Max frontier", "Time (ms)")
    rows = [headers]
    for r in results:
        rows.append((
            r.algorithm,
            "yes" if r.found else "no",
            "-" if r.cost is None else str(r.cost),
            str(r.nodes_expanded),
            str(r.frontier_max_size),
            f"{r.runtime_seconds * 1000:.2f}",
        ))
    widths = [max(len(row[i]) for row in rows) for i in range(len(headers))]
    lines = []
    for i, row in enumerate(rows):
        lines.append(" | ".join(cell.ljust(widths[j]) for j, cell in enumerate(row)))
        if i == 0:
            lines.append("-+-".join("-" * w for w in widths))
    return "\n".join(lines)


def validate_path(grid, path):
    """True if path is a legal walk: free cells, consecutive Manhattan steps of 1."""
    if not path:
        return False
    for pos in path:
        if not grid.is_free(pos):
            return False
    for a, b in zip(path, path[1:]):
        if abs(a[0] - b[0]) + abs(a[1] - b[1]) != 1:
            return False
    return True


def compare_algorithms(grid, start, goal, algorithms=("bfs", "astar")):
    """Run several algorithms on the same start/goal; return SearchResult list."""
    return [search_with_trace(grid, start, goal, algo) for algo in algorithms]


## Demo — load environment and compare BFS vs A*

Uses `data/Environment_Sample.txt` if present, otherwise `notebook_cells/Environment.txt`.
Start = charging station `(0,0)`, goal = D4 on the sample map `(7,0)`.

This cell only calls graph-search helpers above — **no CSP code**.


In [ ]:
# Resolve paths whether the notebook is opened from repo root or notebook_cells/.
candidates = [
    Path("data") / "Environment_Sample.txt",
    Path("notebook_cells") / "Environment.txt",
    Path.cwd() / "data" / "Environment_Sample.txt",
    Path.cwd() / "notebook_cells" / "Environment.txt",
    Path.cwd().parent / "data" / "Environment_Sample.txt",
    Path.cwd().parent / "notebook_cells" / "Environment.txt",
]
env_path = next((p for p in candidates if p.is_file()), None)
if env_path is None:
    raise FileNotFoundError(
        "Could not find Environment_Sample.txt or notebook_cells/Environment.txt"
    )

print(f"Loading: {env_path}")
grid = load_environment(env_path)
print(f"Grid size: {grid.rows} x {grid.cols}")
print(f"Obstacles: {len(grid.obstacles)}")
print(f"Dirty: {grid.dirty_locations}")
print(f"Station: {grid.charging_station}")
print()
print(grid.render_text())
print()

# Sample map: Station (0,0) → D4 (7,0). On other maps, use last dirty as a demo goal.
start = grid.charging_station
goal = (7, 0) if grid.is_in_bounds((7, 0)) and grid.is_free((7, 0)) else grid.dirty_locations[-1]

results = [search_with_trace(grid, start, goal, algo) for algo in ("bfs", "astar")]
print(format_comparison_table(results))
print()

if results[0].path:
    print("BFS path drawn on the grid (* = free cells on the path):")
    print(grid.render_text(path=results[0].path))
    print(
        f"\nOptimal cost: {results[0].cost} "
        f"(Manhattan={heuristic(start, goal)}; "
        f"BFS expanded {results[0].nodes_expanded}, "
        f"A* expanded {results[1].nodes_expanded})"
    )
else:
    print("No path found:", results[0].message)


---
# Part B — CSP Scheduling (Member 2)

Everything below is Ramina's CSP work. It generates a cleaning **order**
(task → time slot). Graph search above finds **paths** between two points.

The two parts are independent:
- Graph search does **not** call CSP.
- CSP may load the same environment file for task labels, but does not need BFS/A*.


# Cleaning Schedule as a CSP

A constraint-satisfaction problem has variables, domains, and rules restricting valid combinations. Here, each dirty location is a task variable (`D1`, `D2`, ...), each time slot is a possible value, precedence rules require one task to come earlier, latest-slot rules set deadlines, and exclusivity prevents two tasks from using the same slot.

Tasks come from `Environment.txt`; slot counts and constraints come from `schedule.txt`. The solver uses MRV with a degree tie-break, forward checking, least-constraining-value ordering, and recursive backtracking.

In [16]:
import sys
from pathlib import Path

workspace_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "core" / "environment.py").is_file()),
    None,
)
if workspace_root is None:
    raise RuntimeError("Could not find the workspace core/environment.py")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from core.environment import load_environment

In [17]:
from typing import Optional

class ScheduleFormatError(Exception):
    """Raised when a schedule file is malformed."""

    def __init__(self, message: str, line_number: Optional[int] = None) -> None:
        self.line_number = line_number
        full = f"Line {line_number}: {message}" if line_number is not None else message
        super().__init__(full)

## Build task variables from the environment

The environment loader provides dirty-location coordinates in file order. We assign labels `D1`, `D2`, and so on from that order, so every dirty location is represented, including tasks absent from the constraints.

In [18]:
#generating variables from the environment

grid=load_environment(workspace_root / "notebook_cells" / "Environment.txt")
variables=[f"D{i+1}" for i in range(len(grid.dirty_locations))]
print(grid)
print(variables)

Grid(rows=10, cols=10, obstacles={(1, 2), (5, 5), (6, 5), (4, 2), (4, 5), (2, 2), (3, 2), (2, 5), (3, 5), (5, 2)}, dirty_locations=[(0, 9), (4, 9), (9, 9), (9, 0), (5, 7)], charging_station=(0, 0))
['D1', 'D2', 'D3', 'D4', 'D5']


## Read the scheduling problem

`load_schedule` reads the slot count, precedence count and pairs, then latest-slot count and pairs. It creates a separate copy of the full slot list for each task. Reading exactly the declared number of pairs supports different constraint counts and avoids depending on a fixed task or slot total.

In [19]:
import re
from pathlib import Path
from typing import Union
def load_schedule(variables,filepath: Union[str, Path]):
    """Load slots and constraints from the count-based schedule file."""
    text = Path(filepath).read_text(encoding="utf-8")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) < 3:
        raise ScheduleFormatError("File too short")

    try:
        n = int(lines[0])
        precedence_count = int(lines[1])
    except ValueError as exc:
        raise ScheduleFormatError("Slot and precedence counts must be integers") from exc
    if n < 1 or precedence_count < 0:
        raise ScheduleFormatError("Slot count must be positive and constraint counts non-negative")

    idx = 2
    precedences = []
    for _ in range(precedence_count):
        if idx >= len(lines):
            raise ScheduleFormatError("File ended before all precedence constraints were read")
        match = re.fullmatch(r"\(\s*(D\d+)\s*,\s*(D\d+)\s*\)", lines[idx])
        if match is None:
            raise ScheduleFormatError(f"Invalid precedence constraint: {lines[idx]!r}")
        precedences.append((match.group(1), match.group(2)))
        idx += 1

    if idx >= len(lines):
        raise ScheduleFormatError("Missing latest-slot constraint count")
    try:
        latest_count = int(lines[idx])
    except ValueError as exc:
        raise ScheduleFormatError("Latest-slot constraint count must be an integer") from exc
    if latest_count < 0:
        raise ScheduleFormatError("Latest-slot constraint count cannot be negative")
    idx += 1

    latest_con = {}
    for _ in range(latest_count):
        if idx >= len(lines):
            raise ScheduleFormatError("File ended before all latest-slot constraints were read")
        match = re.fullmatch(r"\(\s*(D\d+)\s*,\s*T(\d+)\s*\)", lines[idx])
        if match is None:
            raise ScheduleFormatError(f"Invalid latest-slot constraint: {lines[idx]!r}")
        latest_con[match.group(1)] = int(match.group(2))
        idx += 1
    if idx != len(lines):
        raise ScheduleFormatError(f"Unexpected extra schedule input: {lines[idx]!r}")

    slots = list(range(1, n + 1))
    domains = {v: list(slots) for v in variables}

    return domains, precedences, latest_con

The next cell displays the parsed CSP. Each precedence pair is ordered as `(task_that_must_be_earlier, task_that_must_be_later)`. The constraints dictionary uses the key `precedences`, which is the key used by the solver functions.

In [20]:
domains,precedences,latest=load_schedule(variables, workspace_root / "notebook_cells" / "schedule.txt")
print(f"domains: {domains}")
print(f"Precedence: {precedences}")
print(f"latest: {latest}")
constraints={}
constraints["precedences"]=precedences
constraints["latest"]=latest
print(constraints)

domains: {'D1': [1, 2, 3, 4, 5, 6, 7], 'D2': [1, 2, 3, 4, 5, 6, 7], 'D3': [1, 2, 3, 4, 5, 6, 7], 'D4': [1, 2, 3, 4, 5, 6, 7], 'D5': [1, 2, 3, 4, 5, 6, 7]}
Precedence: [('D1', 'D5'), ('D4', 'D1'), ('D3', 'D5'), ('D2', 'D4')]
latest: {'D1': 3, 'D5': 6, 'D3': 4}
{'precedences': [('D1', 'D5'), ('D4', 'D1'), ('D3', 'D5'), ('D2', 'D4')], 'latest': {'D1': 3, 'D5': 6, 'D3': 4}}


## Apply latest-slot limits

A limit such as `D1 by T3` removes slots after 3 from D1's domain, while keeping T3 itself. Tasks without a deadline keep their complete domain.

In [21]:
def apply_latest_slot(domains,latest):
    for task in domains.keys():
        if task in latest:
            
            domains[task]=[x for x in domains[task] if x<= latest[task] ]

    return domains

In [22]:
domains, precedences, latest = load_schedule(variables, workspace_root / "notebook_cells" / "schedule.txt")
apply_latest_slot(domains, latest)

{'D1': [1, 2, 3],
 'D2': [1, 2, 3, 4, 5, 6, 7],
 'D3': [1, 2, 3, 4],
 'D4': [1, 2, 3, 4, 5, 6, 7],
 'D5': [1, 2, 3, 4, 5, 6]}

## Check a candidate assignment

`is_consistent` rejects a value if it misses the task's deadline, reuses an occupied slot, or violates a precedence relation whose other task is already assigned. Precedence is strict: the earlier task's slot number must be smaller.

In [23]:
def is_consistent(variable, value, assignment, constraints):
    latest = constraints["latest"]
    precedences = constraints["precedences"]

    if variable in latest and value > latest[variable]:
        return False

    if value in assignment.values():
        return False

    for before, after in precedences:
        if variable == before and after in assignment:
            if value >= assignment[after]:
                return False
        elif variable == after and before in assignment:
            if value <= assignment[before]:
                return False

    return True

# constraints = {"latest": latest, "precedences": precedences}
# test_cases = [
#     ("D1", 5, {}, False),
#     ("D1", 3, {}, True),
#     ("D4", 3, {"D1": 3}, False),
#     ("D4", 4, {"D1": 3}, False),
#     ("D4", 2, {"D1": 3}, True),
#     ("D5", 2, {"D1": 3}, False),
#     ("D5", 5, {"D1": 3}, True),
# ]

# for variable, value, assignment, expected in test_cases:
#     result = is_consistent(variable, value, assignment, constraints)
#     print(f"{variable}={value}, assignment={assignment}: {result} (expected {expected})")
#     assert result is expected

## Forward checking

After assigning a task, `forward_check` copies the domains so the caller's domains remain unchanged. It removes the chosen slot from other unassigned tasks and narrows precedence neighbors to slots on the correct side. If any remaining task has no possible slot, it returns `None` so search can backtrack.

In [24]:
def forward_check(variable, value, assignment, domains, constraints):
    new_domains = {v: list(d) for v, d in domains.items()}
    precedences = constraints["precedences"]

    for other in new_domains:
        if other == variable or other in assignment:
            continue                      # only unassigned others

        # Rule 1: exclusivity -> 
        if value in new_domains[other]:

            new_domains[other].remove(value)


        # Rule 2: precedence, only for pairs linking variable and other
        for before, after in precedences:
            if before == variable and after == other:
                new_domains[other]=[x for x in new_domains[after] if x>value]

                
            elif after == variable and before == other:
                new_domains[other]=[x for x in new_domains[before] if x<value]
                

        if not new_domains[other]:        # empty domain -> reject
            return None

    return new_domains





### Forward-checking tests

These assertions check exclusivity, both directions of precedence pruning, detection of an empty domain, and that pruning does not mutate the original domains.

In [25]:
step0 = {
    "D1": [1, 2, 3],
    "D2": [1, 2, 3, 4, 5, 6, 7],
    "D3": [1, 2, 3, 4],
    "D4": [1, 2, 3, 4, 5, 6, 7],
    "D5": [1, 2, 3, 4, 5, 6],
}
constraints = {
    "latest": {"D1": 3, "D5": 6, "D3": 4},
    "precedences": [("D1","D5"), ("D4","D1"), ("D3","D5"), ("D2","D4")],
}

# Test 1: D1=3. Exclusivity hits D2/D3, precedence hits D4/D5
r = forward_check("D1", 3, {"D1": 3}, step0, constraints)
assert r["D2"] == [1, 2, 4, 5, 6, 7]
assert r["D3"] == [1, 2, 4]
assert r["D4"] == [1, 2]
assert r["D5"] == [4, 5, 6]
assert r["D1"] == [1, 2, 3]          # assigned variable is not touched

# Test 2: D4=7 leaves D1 nothing (D1 must come after D4) -> failure
assert forward_check("D4", 7, {"D4": 7}, step0, constraints) is None

# Test 3: restore -- the original dictionary must be unchanged
assert step0["D2"] == [1, 2, 3, 4, 5, 6, 7]
assert step0["D5"] == [1, 2, 3, 4, 5, 6]

# Test 4: value missing from some domains must not crash
r = forward_check("D3", 4, {"D3": 4}, step0, constraints)
assert r["D1"] == [1, 2, 3]           # 4 was never there
assert r["D2"] == [1, 2, 3, 5, 6, 7]
assert r["D4"] == [1, 2, 3, 5, 6, 7]
assert r["D5"] == [5, 6]

# Test 5: chained -- feed Test 1's result into the next call
r1 = forward_check("D1", 3, {"D1": 3}, step0, constraints)
r2 = forward_check("D4", 2, {"D1": 3, "D4": 2}, r1, constraints)
assert r2["D2"] == [1]
assert r2["D3"] == [1, 4]
assert r2["D5"] == [4, 5, 6]

# Test 6: emptiness from exclusivity alone, no precedence involved
tiny = {"A": [1], "B": [1]}
assert forward_check("A", 1, {"A": 1}, tiny, {"latest": {}, "precedences": []}) is None

print("all forward_check tests passed")

all forward_check tests passed


## Select the next task: MRV and degree

The Minimum Remaining Values (MRV) heuristic chooses an unassigned task with the fewest legal slots, which exposes failures early. If tasks tie, the degree heuristic chooses the one connected to the most still-unassigned tasks by precedence constraints. The task label provides a deterministic final tie-break.

In [26]:
def select_unassigned_variable(assignment, domains, constraints):
    unassigned = [task for task in domains if task not in assignment]
    if not unassigned:
        return None

    smallest = min(len(domains[task]) for task in unassigned)
    tied = [task for task in unassigned if len(domains[task]) == smallest]
    return min(tied, key=lambda task: (-degree(task, assignment, constraints), int(task[1:])))



def degree(task, assignment, constraints):
    count = 0
    for before, after in constraints["precedences"]:
        if before == task and after not in assignment:
            count += 1
        elif after == task and before not in assignment:
            count += 1
    return count

            

            
    




In [27]:
print(degree)                                          # should say: <function degree ...>
print(degree("D1", {}, constraints))                   # should print 2
print(select_unassigned_variable.__code__.co_varnames) # should include 'tied'

<function degree at 0x00000138280EC4A0>
2
('assignment', 'domains', 'constraints', 'unassigned', 'tied')


## Order candidate slots: LCV

The optional Least Constraining Value heuristic tries slots that remove the fewest choices from other tasks first. `forward_check` predicts the pruning for each candidate; choices causing an empty domain are ranked last. The numeric slot breaks ties consistently.

In [28]:
def order_domain_values(variable, assignment, domains, constraints):
    def cost(value):
        result = forward_check(variable, value, assignment, domains, constraints)
        if result is None:
            return float("inf")
        removed = sum(
            len(domains[task]) - len(result[task])
            for task in domains
            if task != variable
        )
        return removed
    return sorted(domains[variable], key=lambda v: (cost(v), v))

## Backtracking search

The recursive solver selects a task, tries candidate slots in LCV order, checks consistency, then forward-checks the remaining domains. If a branch fails, it removes that task's tentative assignment and tries the next value. Since each forward-check call returns copied domains, earlier domain lists are restored naturally when recursion returns.

In [29]:
def backtracking_search(variables, domains, constraints):
    def backtrack(assignment, domains):
        if len(assignment) == len(variables):
            return assignment
        var = select_unassigned_variable(assignment, domains, constraints)
        for value in order_domain_values(var, assignment, domains, constraints):
            print("  " * len(assignment), var, "=", value)
            if is_consistent(var, value, assignment, constraints):
                assignment[var] = value
                new_domains = forward_check(var, value, assignment, domains, constraints)
                if new_domains is not None:
                    result = backtrack(assignment, new_domains)
                    if result is not None:
                        return result
                del assignment[var]            # undo
        return None
    return backtrack({}, {v: list(domains[v]) for v in variables})

## Run and interpret the result

The main cell reloads the environment and schedule file, applies the latest-slot limits, assembles the constraints, and calls the solver. A returned dictionary maps every task to its slot; `None` means no feasible schedule was found. The printed schedule lists tasks in time order. With the current inputs, the expected result is `T1: D2`, `T2: D4`, `T3: D1`, `T4: D3`, and `T5: D5`; T6 and T7 are unused.

In [30]:
def main():
    environment_path = workspace_root / "notebook_cells" / "Environment.txt"
    schedule_path = workspace_root / "notebook_cells" / "schedule.txt"

    grid = load_environment(environment_path)
    variables = [f"D{i + 1}" for i in range(len(grid.dirty_locations))]
    domains, precedences, latest = load_schedule(variables, schedule_path)
    domains = apply_latest_slot(domains, latest)
    constraints = {"precedences": precedences, "latest": latest}

    schedule = backtracking_search(variables, domains, constraints)
    if schedule is None:
        print("No feasible schedule exists for schedule.txt.")
        return None

    print("Generated schedule:")
    for task, slot in sorted(schedule.items(), key=lambda item: item[1]):
        print(f"T{slot}: {task}")
    return schedule


if __name__ == "__main__":
    schedule = main()

 D1 = 2
   D4 = 1
 D1 = 3
   D4 = 2
     D2 = 1
       D3 = 4
         D5 = 5
Generated schedule:
T1: D2
T2: D4
T3: D1
T4: D3
T5: D5
